## 1. Import Libraries

In this section i import the libraries needed for the project:
- 'pathlib.Path': for file Path handling
- 'pandas': to load and manipulate dataset
- 'scikit-learn': for train/validation/test split
- 'collections.Counter': to check class balance
- 'transformers': to load the pretrained BERT model and tokenizer
- 'transformers.BertTokenizer': to convert text into tokens BERT can process
- 'transformers.BertForSequenceClassification': the pretrained BERT model with a classification head, used for fine-tuning on 3-classes task
- 'torch': for tensor operations and GPU support

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split
from collections import Counter

from transformers import BertTokenizer, BertForSequenceClassification
import torch

In [ ]:
BASE_DIR = Path.cwd().parent

## 1.5 Function

- 'get_token_count': takes a single phrase as input, tokenizes it using the BERT tokenizer, and returns the number of tokens it was split into. 

In [ ]:
def get_token_count(text):
    tokens = tokenizer.encode(text) # tokenize the input text using BERT's tokenizer
    return len(tokens) # return how many tokens the phrase was split into


## 2.  Import Dataset

I load the CSV file containing the labeled dataset (390 phrases split across the three classes: SOS, MAINTENANCE, SERVICE) and preview the first few rows to confirm it loaded correctly.

In [ ]:
df_Path = BASE_DIR / "data"/"raw"/"skyguard_dataset.csv"

df = pd.read_csv(df_Path)

df.head()

## 3. Dataset Quality Check

Before splitting the data, I check:
- the number of samples per class (to confirm balance)
- the presence of duplicate or near-duplicate rows
- any missing/empty values

In [ ]:
print("Class distribution:")
print(Counter(df["label"]))

print("\nMissing values:")
print(df.isnull().sum())

print("\nExact duplicate rows:", df.duplicated().sum())
print("Duplicate texts only:", df["text"].duplicated().sum())

## 4. Train/Validation/Test split

I split the dataset into three parts:
- **Train**: used to fine-tune the model
- **Validation**: used to monitor the performance during training
- **Test**: held out completely, used for the final evaluation of the model

The split is stratified by label, so each subset keeps the same class proportions as the original dataset.

In [ ]:
train_df, temp_df = train_test_split( 
    df,
    test_size= 0.30,       # First split: Separate 70% of data for training (train_df) and 30% for a temporary set (temp_df)
    stratify= df['label'], # stratify keeps the class proportions identical to the original df
    random_state= 42       # random_state sets a fixed seed to guarantee reproducible splits
       )


val_df, test_df = train_test_split(  
    temp_df,               
    test_size= 0.50,            # Second split: Divide the temporary set equally (50/50) into validation (val_df) and test (test_df) sets      
    stratify= temp_df['label'], # This results in exactly 15% validation and 15% test of the total dataset (df)
    random_state= 42
)

print("Train size: ", len(train_df))
print("Validation size: ", len(val_df))
print("Test size: ", len(test_df))

print('\n')

print("Train class distribution: ", Counter(train_df['label']))
print("Validation class distribution: ", Counter(val_df['label']))
print("Test class distribution: ", Counter(test_df['label']))

## 5. Save splits to disk

I save the train, validation, and test sets as separate CSV files in `data/processed/`, so they can be reused without recomputing the split.

In [ ]:
train_df.to_csv(BASE_DIR / "data" / "processed" / "train.csv", index= False)
val_df.to_csv(BASE_DIR / "data" / "processed" / "val.csv", index= False)
test_df.to_csv(BASE_DIR / "data" / "processed" / "test.csv", index = False)

print("Files saved successfully in 'data/processed/'")

## 6. Load Pretrained Model and Tokenizer

I load 'bert-base-uncased' from Hugging Face, along with its tokenizer.

Since  I have 3 classes (SOS, MAINTENANCE; SERVICE), I configure the model with 'num_label = 3' and a mapping between labels and their numeric IDs.

In [ ]:
label2id = {"SOS" : 0, "MAINTENANCE" : 1, "SERVICE" : 2} # maps each class name to a numeric ID
id2label = {0 : "SOS", 1 : "MAINTENANCE", 2 : "SERVICE"} # reverse mapping, numeric ID back to class name

model_name = "bert-base-uncased" # pretrained model to fine-tune

tokenizer = BertTokenizer.from_pretrained(model_name)  # loads the tokenizer matching bert-base-uncased
model = BertForSequenceClassification.from_pretrained(
    model_name,
    num_labels = 3, # 3 output classes: SOS, MAINTENANCE, SERVICE
    label2id = label2id, # required by Hugging Face to map labels to IDs
    id2label = id2label  # required by Hugging Face to map IDs back to labels
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # use GPU if available, otherwise fall back to CPU

# Explicit equivalent of the inline 'if' statement above for clarity:
# if torch.cuda.is_available():
#     device = torch.device("cuda")
# else:
#     device = torch.device("cpu")

model.to(device) # move model weights to the selected device

print("Model loaded on: ", device)

## 7. Tokenization

Before tokenizing the full dataset, I check the token lenght distribution of the phrases to choose an appropriate 'max_length' value (avoiding unnecessary padding or truncation)

Then I tokenize the train, validation and test sets using the BERT tokenizer, converting each phrase into input IDs and attention masks the model can process.

In [ ]:
token_lengths = df["text"].apply(get_token_count)  # apply the 'get_token_count' function to every phrase in the dataset


print("Max token length:", token_lengths.max())
print("Mean token length:", token_lengths.mean())
print("95th percentile:", token_lengths.quantile(0.95))

Based on the token length analysis above (max 38, mean ~22, 95th percentile 28), I set 'max_length=50': a round number that comfortably covers the longest phrase in the dataset while keeping tokenization lightweight.

In [ ]:
MAX_LENGTH = 50 # chosen based on the token length analysis before

train_encodings = tokenizer(
    list(train_df["text"]),
    truncation = True,
    padding = "max_length",
    max_length = MAX_LENGTH,
    return_tensors = "pt"
)

val_encodings = tokenizer(
    list(val_df["text"]),
    truncation = True,
    padding = "max_length",
    max_length = MAX_LENGTH,
    return_tensors = "pt"
)

test_encodings = tokenizer(
    list(test_df["text"]),
    truncation = True,
    padding = "max_length",
    max_length = MAX_LENGTH,
    return_tensors = "pt"
)


print("Train encodings shape:", train_encodings["input_ids"].shape)
print("Validation encodings shape:", val_encodings["input_ids"].shape)
print("Test encodings shape:", test_encodings["input_ids"].shape)